# Star Wars Knowledge Graph Agent System

An intelligent agent system for querying and exploring the Star Wars universe through RDF/SPARQL.
This notebook demonstrates the full agent system in action.

## Section 1: Import Required Libraries

In [ ]:
import sys
from pathlib import Path
import json
from pprint import pprint

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

# Import our modules
from src.rdf_graph_loader import SWAPIGraphDB, initialize_db
from src.agent import StarWarsGraphAgent
from src.query_builder import SPARQLQueryBuilder

print("✅ All imports successful!")

## Section 2: Load and Parse RDF Data

In [ ]:
# Load the RDF data
ttl_path = Path("..") / "SWAPI-WD-data.ttl"
print(f"Loading RDF from: {ttl_path.absolute()}")

db = initialize_db(str(ttl_path))
print(f"✅ Graph loaded successfully!")

### Explore the Graph Structure

In [ ]:
# Get graph statistics
stats = db.get_graph_stats()
print("📊 Graph Statistics:")
print(f"  Total Triples: {stats['total_triples']:,}")
print(f"  Total Entities: {stats['total_entities']:,}")
print(f"  Total Relations: {stats['total_relations']:,}")

print("\n📋 Entity Type Breakdown:")
for entity_type, count in sorted(stats['entity_types'].items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {entity_type}: {count}")

### View Graph Schema

In [ ]:
# Get schema information
schema = db.get_graph_schema()

print(f"📚 Graph Schema\n")
print(f"Classes ({len(schema['classes'])}):")
for cls in sorted(schema['classes'])[:15]:
    print(f"  - {cls}")

print(f"\nProperties ({len(schema['properties'])}):")
for prop in sorted(schema['properties'])[:15]:
    print(f"  - {prop}")

## Section 3: Direct Graph Queries (SPARQL)

### Query 1: Count Characters by Type

In [ ]:
# Count all characters
query = """
PREFIX voc: <https://swapi.co/vocabulary/>
SELECT (COUNT(?char) as ?count) WHERE {
    ?char a voc:Character .
}
"""

results = db.execute_sparql(query)
print("Total Characters in Database:")
pprint(results)

### Query 2: Find Character by Name

In [ ]:
# Find Luke Skywalker and his properties
query = """
PREFIX voc: <https://swapi.co/vocabulary/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?entity ?property ?value WHERE {
    ?entity rdfs:label ?label ;
            ?property ?value .
    FILTER(CONTAINS(LCASE(str(?label)), "luke"))
}
LIMIT 20
"""

results = db.execute_sparql(query)
print(f"Found {len(results)} properties for characters with 'Luke':")
for result in results[:10]:
    print(f"  Property: {str(result['property']).split('/')[-1]:20} = {result['value']}")

### Query 3: Character Homeworlds

In [ ]:
# Find characters and their homeworlds
query = """
PREFIX voc: <https://swapi.co/vocabulary/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?charLabel ?worldLabel WHERE {
    ?char rdfs:label ?charLabel ;
          voc:homeworld ?world .
    ?world rdfs:label ?worldLabel .
}
LIMIT 15
"""

results = db.execute_sparql(query)
print("Characters and their Homeworlds:")
for result in results:
    print(f"  {str(result['charLabel']):30} → {result['worldLabel']}")

### Query 4: Average Height by Species

In [ ]:
# Get average height of characters with height data
query = """
PREFIX voc: <https://swapi.co/vocabulary/>
SELECT (AVG(?height) as ?avg_height) (MIN(?height) as ?min_height) (MAX(?height) as ?max_height) WHERE {
    ?char voc:height ?height .
    FILTER(?height > 0)
}
"""

results = db.execute_sparql(query)
print("Height Statistics (for characters with recorded heights):")
if results:
    r = results[0]
    print(f"  Average Height: {float(r['avg_height']):.1f} cm")
    print(f"  Minimum Height: {float(r['min_height']):.1f} cm")
    print(f"  Maximum Height: {float(r['max_height']):.1f} cm")

## Section 4: Initialize the AI Agent

In [ ]:
# Initialize the AI agent
print("Initializing AI Agent...")
agent = StarWarsGraphAgent(db, model="gpt-4")
print("✅ Agent initialized!")

# Show system prompt info
print("\n📋 Agent has knowledge of:")
schema = agent.graph_schema
print(f"  - {len(schema['classes'])} entity classes")
print(f"  - {len(schema['properties'])} properties")
print(f"  - {agent.graph_stats['total_triples']:,} RDF triples")

## Section 5: Define Query Tools (Using Query Builder)

In [ ]:
# Initialize query builder
builder = SPARQLQueryBuilder()

# Show available templates
print("📚 Available Query Templates:\n")
templates = builder.get_template_info()
for i, template in enumerate(templates, 1):
    print(f"{i}. {template['name']}")
    print(f"   Description: {template['description']}")
    print(f"   Example: {template['example']}")
    print()

### Build Sample Queries

In [ ]:
# Example 1: Find by name
query = builder.build_find_by_name("Luke")
print("Query: Find characters named Luke")
print("-" * 60)
print(query)
print("-" * 60)

results = db.execute_sparql(query)
print(f"\nResults: {len(results)} found")
for result in results[:3]:
    print(f"  - {result}")

In [ ]:
# Example 2: Count by type
query = builder.build_count_query("Droid")
print("Query: Count all Droid characters")
print("-" * 60)
print(query)
print("-" * 60)

results = db.execute_sparql(query)
print(f"\nResults:")
pprint(results)

## Section 6: Natural Language Queries with Agent

### Query 1: Character Information

In [ ]:
# Natural language query 1
question = "Who is Yoda and what do we know about him?"
print(f"❓ Question: {question}\n")

try:
    sparql_query, results = agent.query(question)
    
    print("🔍 Generated SPARQL Query:")
    print("-" * 60)
    print(sparql_query)
    print("-" * 60)
    
    print(f"\n✅ Results ({len(results)} found):\n")
    for i, result in enumerate(results[:5], 1):
        print(f"{i}. {result}")
except Exception as e:
    print(f"⚠️  Note: Full agent requires OpenAI API key. Error: {e}")

### Query 2: Statistics

In [ ]:
# Natural language query 2
question = "How many characters are there in total?"
print(f"❓ Question: {question}\n")

try:
    sparql_query, results = agent.query(question)
    
    print("🔍 Generated SPARQL Query:")
    print("-" * 60)
    print(sparql_query)
    print("-" * 60)
    
    print(f"\n✅ Results:\n")
    pprint(results)
except Exception as e:
    print(f"⚠️  Note: Full agent requires OpenAI API key. Error: {e}")

### Query 3: Relationships

In [ ]:
# Natural language query 3
question = "Which characters are from Tatooine?"
print(f"❓ Question: {question}\n")

try:
    sparql_query, results = agent.query(question)
    
    print("🔍 Generated SPARQL Query:")
    print("-" * 60)
    print(sparql_query)
    print("-" * 60)
    
    print(f"\n✅ Results ({len(results)} found):\n")
    for i, result in enumerate(results[:10], 1):
        print(f"{i}. {result}")
except Exception as e:
    print(f"⚠️  Note: Full agent requires OpenAI API key. Error: {e}")

## Section 7: Advanced Features

### Entity Properties Lookup

In [ ]:
# Get all properties of a specific entity
# First, find an entity
results = db.find_by_label("Luke")

if results:
    entity_uri, label = results[0]
    print(f"Entity: {label}")
    print(f"URI: {entity_uri}")
    print("\nProperties:")
    
    props = db.get_entity_properties(entity_uri)
    for prop, value in sorted(props.items()):
        if isinstance(value, list):
            print(f"  {prop}: {value}")
        else:
            print(f"  {prop}: {value}")

### Conversation History

In [ ]:
# View conversation history
history = agent.get_conversation_history()

print(f"Conversation History ({len(history)} messages):\n")
for i, msg in enumerate(history, 1):
    role = "👤 You" if msg['role'] == 'user' else "🤖 Agent"
    content = msg['content'][:100] + "..." if len(msg['content']) > 100 else msg['content']
    print(f"{i}. {role}: {content}")

## Section 8: Summary and Next Steps

In [ ]:
print("""\n🎉 SUMMARY
================

You've successfully explored the Star Wars Knowledge Graph Agent System!

✅ What you learned:
   1. How to load and parse RDF data with rdflib
   2. How to execute SPARQL queries
   3. How to use the query builder with templates
   4. How to initialize and use the AI agent
   5. How to convert natural language to SPARQL queries
   6. How to explore graph schema and entity properties

🚀 Next steps:
   1. Run 'python main.py' from the terminal for the interactive chat
   2. Explore more complex SPARQL queries
   3. Integrate with your own applications using the Python API
   4. Check README.md for advanced features and examples

💡 Example API usage:
   from src.rdf_graph_loader import initialize_db
   from src.agent import StarWarsGraphAgent
   
   db = initialize_db("SWAPI-WD-data.ttl")
   agent = StarWarsGraphAgent(db)
   sparql, results = agent.query("Who is your question?")

📚 Available modules:
   - src.rdf_graph_loader: Core RDF/graph operations
   - src.query_builder: SPARQL template generation
   - src.agent: AI agent for NL→SPARQL conversion
   - src.chat_interface: Interactive CLI chat
""")